# run LLMs in transformers

In [1]:
# check dependencies

import transformers
print('transformers:', transformers.__version__)

import torch
print('pytorch', torch.__version__)
print('cuda:', torch.version.cuda, torch.cuda.is_available())

import torchvision
print('torchvision:', torchvision.__version__)
import torchaudio
print('torchaudio:', torchaudio.__version__)

/home/yuan/anaconda3/envs/agent2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 5.12.1
pytorch 2.11.0+cu130
cuda: 13.0 True
torchvision: 0.26.0+cu130
torchaudio: 2.11.0+cu130


## test hugging face

In [12]:
# load model and toknenizer
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-7B-Instruct"

model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 339/339 [00:00<00:00, 19677.94it/s]


In [13]:
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
model_inputs

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,  35127,    752,    264,
           2805,  16800,    311,   3460,   4128,   1614,     13, 151645,    198,
         151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [14]:
# generate contents
generated_ids = model.generate(**model_inputs, max_new_tokens=512)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
response

'Certainly! A Large Language Model (LLM) is a type of artificial intelligence model designed to understand and generate human-like text based on the input it receives. These models are typically trained on vast amounts of text data from the internet, books, articles, and other sources to learn patterns, semantics, and syntax in natural language.\n\nKey characteristics of LLMs include:\n\n1. **Scalability**: They are often very large, containing millions or even billions of parameters, which allows them to capture complex linguistic structures and nuances.\n2. **Context Understanding**: LLMs can understand and respond to questions and statements with context, making their interactions more natural and useful.\n3. **Versatility**: They can be used for various natural language processing tasks such as text completion, translation, summarization, question answering, and more.\n4. **Continuous Learning**: Many LLMs are fine-tuned on specific datasets to enhance their performance on particul

## access LLM

In [5]:
from dotenv import load_dotenv

load_dotenv(dotenv_path='/home/yuan/bio/yuan/.env')

True

In [8]:
# build vector store locally
import os
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

# 1. Set up Ollama embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key = os.getenv("OPEN_ROUTER_API_KEY"),
    base_url = "https://openrouter.ai/api/v1",
    check_embedding_ctx_length=False,
    model_kwargs={"encoding_format": "float"}

)
# 2. Load and chunk your documents
loader = TextLoader('/home/yuan/bio/drug_screening_ai/data/pdf_layout/PMC2893758/PMC2893758.txt')
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
)
chunks = text_splitter.split_documents(documents)

# 3. Build FAISS index
db = FAISS.from_documents(chunks, embeddings)

# 4. Save the index for later use
db.save_local("faiss_index")

# 5. Later, load and query
db = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True,
)

# Query
results = db.similarity_search("Your question here", k=4)

In [3]:
import os
from langchain_openai import OpenAIEmbeddings

# Set the environment variable BEFORE initializing
# os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_ROUTER_API_KEY")
# os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

# Now initialize without passing parameters explicitly
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    api_key=os.getenv("OPEN_ROUTER_API_KEY"),
    base_url = "https://openrouter.ai/api/v1",
    check_embedding_ctx_length=False,
    model_kwargs={"encoding_format": "float"}
)

# Test it
try:
    vector = embeddings.embed_query("Hello world")
    print(f"✅ Success! Vector length: {len(vector)}")
except Exception as e:
    print(f"❌ Error: {e}")

/home/yuan/anaconda3/envs/agent2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Success! Vector length: 1536


In [43]:
import os
from langchain_openai import OpenAIEmbeddings

# Set the environment variable BEFORE initializing
os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_ROUTER_API_KEY")
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

# Now initialize without passing parameters explicitly
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    check_embedding_ctx_length=False,
    model_kwargs={"encoding_format": "float"}
)

# Test it
try:
    vector = embeddings.embed_query("Hello world")
    print(f"✅ Success! Vector length: {len(vector)}")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Success! Vector length: 1536


In [35]:
# directy OpenAI client using OpenRouter
import os
from openai import OpenAI

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv('OPEN_ROUTER_API'), 
)

model_name = "qwen/qwen-2.5-7b-instruct"
prompt = [{"role": "user", "content": "Write a short poem about coding."}]
completion = client.chat.completions.create(model=model_name, messages=prompt)

print(completion.choices[0].message.content)

In lines of syntax and logic so tight,
A dance of zeros and ones through the night.
Syntax and semicolons in perfect array,
 Coding constructs a virtual ballet every day.

Functions and classes take shape in our hands,
A digital tale, dramas and landscapes transcend.
Syntax whispers secrets in binary code,
Each keystroke a brushstroke on the digital ode.

Loops and conditionals guide each twist and turn,
Through darkness and light, searching for a numeric yearn.
Errors rise, challenges meet head-on,
Yet with perseverance and skill, the code finds its home.

In the end, a program runs, a masterpiece born,
From lines of code, a digital life, a work of art earned.


In [36]:
# langchain + open router
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenRouter(
    model="qwen/qwen-2.5-7b-instruct",
    api_key=os.getenv('OPEN_ROUTER_API'), 
    temperature=0,
    max_tokens=512
)

# Simple invocation
response = llm.invoke("What is machine learning?")
print(response.content)

# With messages
messages = [
    SystemMessage(content="You are a helpful assistant"),
    HumanMessage(content="What is AI?")
]
response = llm.invoke(messages)

# Structured output
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int

structured_llm = llm.with_structured_output(Person)
result = structured_llm.invoke("John is 30 years old")

Machine learning is a subset of artificial intelligence (AI) that involves the development of algorithms and statistical models that enable computer systems to improve their performance on a specific task through experience or data. Essentially, machine learning allows software applications to become more accurate in predicting outcomes without being explicitly programmed to perform the task.

Key aspects of machine learning include:

1. **Data**: Machine learning algorithms require large amounts of data to learn from. The quality and quantity of data are crucial for the effectiveness of the model.

2. **Algorithms**: These are the mathematical models that the machine learning system uses to learn from the data. Common types include decision trees, neural networks, support vector machines, and clustering algorithms.

3. **Training**: The process of using the algorithm to learn from the data. During training, the algorithm adjusts its parameters to minimize errors in predictions.

4. **